YOLOv11 香菸檢測優化訓練 - A100 GPU專用版本
針對極小物體檢測的全面優化配置

In [ ]:
from google.colab import drive
import os
import sys
import shutil
import time
import subprocess
import warnings
import numpy as np
import torch
from pathlib import Path
from typing import Optional, Dict, Any, Tuple, List
from PIL import Image
import matplotlib.pyplot as plt
import cv2
from glob import glob

# 設置警告過濾

In [ ]:
warnings.filterwarnings('ignore')
drive.mount('/content/drive')

class SmokingDetectionConfigV11:
    """專門針對YOLOv11香菸檢測的配置類別"""

    # 基本路徑設定
    DRIVE_BASE = Path("/content/drive/MyDrive/yolo_workspace")
    DATASET_ZIP = "train_dataset_v6.zip"
    DATASET_PATH = Path("/content/train_dataset_v6")

    # 實驗設定
    EXPERIMENT_NAME = "smoking_detect_v11m_ultra"
    BASE_MODEL = "yolo11m.pt"  # YOLOv11 nano版本，對小物體更敏感
    PROJECT_PATH = DRIVE_BASE / "yolo11_output"

    # YOLOv11 A100 GPU 優化參數
    EPOCHS = 140  # YOLOv11收斂更慢但效果更好
    IMG_SIZE = 960  # YOLOv11支持更高解析度且效率更好
    BATCH_SIZE = 48  # YOLOv11記憶體效率提升

    # YOLOv11新增的小物體檢測特化參數
    CONF_THRESHOLD = 0.15  # YOLOv11對低置信度檢測更準確
    IOU_THRESHOLD = 0.45   # 更精細的NMS
    MAX_DETECTIONS = 500   # YOLOv11可處理更多檢測

    # 多尺度訓練參數 - YOLOv11優化版
    MULTISCALE_RANGE = (0.3, 2.5)  # 更廣的尺度範圍

    @property
    def data_yaml_path(self) -> Path:
        return self.DATASET_PATH / "data.yaml"

    @property
    def best_model_path(self) -> Path:
        return self.PROJECT_PATH / self.EXPERIMENT_NAME / "weights" / "best.pt"

config = SmokingDetectionConfigV11()

Mounted at /content/drive


In [ ]:
class YOLOv11SmokingTrainer:
    """專門針對YOLOv11和香菸檢測優化的訓練器"""

    def __init__(self, config: SmokingDetectionConfigV11):
        self.config = config
        self.model = None
        self.device_info = self._get_device_info()

    def _get_device_info(self) -> Dict[str, Any]:
        """獲取GPU信息並優化設置"""
        info = {
            'device': 'cuda' if torch.cuda.is_available() else 'cpu',
            'gpu_count': torch.cuda.device_count(),
            'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
            'memory_gb': torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
        }

        print(f"🚀 GPU資訊: {info['gpu_name']}")
        print(f"💾 GPU記憶體: {info['memory_gb']:.1f} GB")

        # A100特殊優化 + YOLOv11優化
        if 'A100' in info['gpu_name']:
            print("⚡ 檢測到A100 GPU，啟用YOLOv11專用優化")
            self.config.BATCH_SIZE = min(64, self.config.BATCH_SIZE * 1.5)

        return info

    def setup_environment(self) -> bool:
        """設置YOLOv11訓練環境"""
        print("\n🛠️ 設置YOLOv11優化環境...")

        # 安裝YOLOv11
        os.system("pip install -q ultralytics>=8.3.0")  # 確保支持YOLOv11

        # YOLOv11專用GPU優化設置
        if torch.cuda.is_available():
            torch.backends.cudnn.benchmark = True
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
            # YOLOv11新增優化
            torch.backends.cudnn.deterministic = False
            torch.backends.cudnn.enabled = True

        # 解壓數據集
        zip_path = self.config.DRIVE_BASE / self.config.DATASET_ZIP
        if zip_path.exists():
            os.system(f'unzip -q "{zip_path}" -d /content/')
            print("✅ 數據集解壓完成")
        else:
            print(f"❌ 找不到數據集: {zip_path}")
            return False

        return self._validate_dataset()

    def _validate_dataset(self) -> bool:
        """驗證數據集並分析小物體分佈"""
        if not self.config.data_yaml_path.exists():
            print("❌ data.yaml 不存在")
            return False

        self._analyze_small_objects()
        return True

    def _analyze_small_objects(self) -> None:
        """分析數據集中小物體的分佈"""
        print("\n🔍 分析小物體分佈...")

        labels_dir = self.config.DATASET_PATH / "train" / "labels"
        if not labels_dir.exists():
            return

        box_sizes = []
        for label_file in labels_dir.glob("*.txt"):
            with open(label_file, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        w, h = float(parts[3]), float(parts[4])
                        box_sizes.append(w * h)

        if box_sizes:
            box_sizes = np.array(box_sizes)
            print(f"📊 物體大小統計:")
            print(f"   平均大小: {box_sizes.mean():.6f}")
            print(f"   最小大小: {box_sizes.min():.6f}")
            print(f"   最大大小: {box_sizes.max():.6f}")
            print(f"   小於0.01的比例: {(box_sizes < 0.01).mean():.2%}")
            print(f"   小於0.001的比例: {(box_sizes < 0.001).mean():.2%}")

    def train(self) -> Optional[Any]:
        """執行YOLOv11優化訓練"""
        from ultralytics import YOLO

        print(f"\n🎯 開始YOLOv11優化訓練 - {self.config.EXPERIMENT_NAME}")

        start_time = time.time()

        # 載入YOLOv11模型
        self.model = YOLO(self.config.BASE_MODEL)

        # YOLOv11專用訓練參數 - 針對小物體優化
        train_args = {
            'data': str(self.config.data_yaml_path),
            'epochs': self.config.EPOCHS,
            'imgsz': self.config.IMG_SIZE,
            'batch': self.config.BATCH_SIZE,

            # === YOLOv11 新增/改進的小物體檢測優化 ===

            # 1. 多尺度訓練增強 (YOLOv11改進版)
            'scale': 0.7,          # YOLOv11優化的尺度增強
            'mosaic': 1.0,         # 保持最大mosaic增強
            'mixup': 0.15,         # YOLOv11調整後的mixup
            'copy_paste': 0.3,     # YOLOv11增強的copy-paste

            # 2. YOLOv11新增的小物體特化參數
            'rect': True,          # 矩形訓練，提高小物體檢測效率
            #'mosaic_prob': 0.8,    # YOLOv11新增：mosaic概率控制
            #'mixup_prob': 0.1,     # YOLOv11新增：mixup概率控制

            # 3. 幾何增強 - YOLOv11優化版
            'degrees': 3,          # 減少旋轉避免小物體變形
            'translate': 0.03,     # YOLOv11優化的平移
            'shear': 0.5,         # YOLOv11調整的剪切
            'perspective': 0.0002, # YOLOv11新增：透視變換
            'flipud': 0.0,        # 香菸檢測不需要上下翻轉
            'fliplr': 0.5,        # 保持左右翻轉

            # 4. 顏色增強 - 針對香菸檢測優化
            'hsv_h': 0.003,       # YOLOv11微調色調
            'hsv_s': 0.2,         # 飽和度調整
            'hsv_v': 0.15,        # 亮度調整
            'auto_augment': 'randaugment',  # YOLOv11自動增強

            # 5. YOLOv11學習率策略優化
            'lr0': 0.001,         # YOLOv11推薦初始學習率
            'lrf': 0.0001,        # 最終學習率
            'momentum': 0.937,    # YOLOv11優化動量
            'weight_decay': 0.0005, # 權重衰減
            'warmup_epochs': 3,   # YOLOv11預熱輪次
            'warmup_momentum': 0.8,
            'warmup_bias_lr': 0.1,
            'cos_lr': True,       # 餘弦學習率衰減

            # 6. YOLOv11損失函數優化 - 專門針對小物體
            'box': 7.5,           # YOLOv11建議的box loss權重
            'cls': 0.5,           # 分類權重
            'dfl': 1.5,           # DFL loss權重
            'pose': 12.0,         # YOLOv11新增
            'kobj': 1.0,          # YOLOv11新增

            # 7. YOLOv11架構特定參數
            'dropout': 0.0,       # YOLOv11建議不使用dropout
            #'val_period': 5,      # 驗證頻率

            # 8. A100硬體優化
            'amp': True,          # 混合精度訓練
            'fraction': 1.0,      # 使用全部數據
            'cache': 'ram',       # 數據緩存到RAM
            'workers': 16,        # A100可處理更多worker
            'device': '0',        # 指定GPU

            # 9. YOLOv11檢測優化
            'conf': self.config.CONF_THRESHOLD,
            'iou': self.config.IOU_THRESHOLD,
            'max_det': self.config.MAX_DETECTIONS,
            'agnostic_nms': False,  # YOLOv11建議
            'retina_masks': True,   # YOLOv11高質量mask

            # 10. 訓練策略
            'patience': 50,      # YOLOv11更大耐心值
            'save': True,
            'save_period': 20,    # 保存頻率
            'plots': True,
            'val': True,
            'split': 'val',       # YOLOv11驗證分割

            # 11. 項目設置
            'project': str(self.config.PROJECT_PATH),
            'name': self.config.EXPERIMENT_NAME,
            'exist_ok': True,
            'verbose': True,
            'seed': 42,           # 設置隨機種子確保可重現
            'deterministic': False, # YOL�v11性能優化
        }

        try:
            # 執行訓練
            results = self.model.train(**train_args)

            train_time = time.time() - start_time
            print(f"\n⏰ 訓練完成，用時 {train_time/60:.2f} 分鐘")

            # 訓練結果分析
            self._analyze_training_results(results)

            return results

        except Exception as e:
            print(f"❌ 訓練失敗: {e}")
            import traceback
            traceback.print_exc()
            return None

    def _analyze_training_results(self, results) -> None:
        """分析YOLOv11訓練結果"""
        print(f"\n📈 YOLOv11訓練結果分析:")

        if hasattr(results, 'results_dict'):
            rd = results.results_dict
            print(f"   🎯 最佳mAP50: {rd.get('metrics/mAP50(B)', 'N/A')}")
            print(f"   🎯 最佳mAP50-95: {rd.get('metrics/mAP50-95(B)', 'N/A')}")

            # 小物體特殊指標
            if 'metrics/mAP50(S)' in rd:
                print(f"   🔍 小物體mAP50: {rd.get('metrics/mAP50(S)', 'N/A')}")

    def validate_model(self) -> Dict[str, float]:
        """驗證YOLOv11模型，特別關注小物體檢測性能"""
        from ultralytics import YOLO

        if not self.config.best_model_path.exists():
            print("❌ 找不到最佳模型")
            return {}

        print("\n🔍 驗證YOLOv11模型性能...")

        model = YOLO(str(self.config.best_model_path))

        # 使用不同置信度閾值測試
        thresholds = [0.05, 0.1, 0.15, 0.25, 0.5]
        results = {}

        for conf in thresholds:
            val_results = model.val(
                conf=conf,
                iou=self.config.IOU_THRESHOLD,
                max_det=self.config.MAX_DETECTIONS,
                split='val'
            )
            results[f'mAP50_conf_{conf}'] = val_results.box.map50
            print(f"   置信度{conf}: mAP50={val_results.box.map50:.3f}")

        return results

def install_yolov11_dependencies():
    """安裝YOLOv11所需依賴"""
    print("📦 安裝YOLOv11必要套件...")

    packages = [
        "ultralytics>=8.3.0",  # 確保支持YOLOv11
        "albumentations",
        "onnx>=1.16.0",        # 更新版本
        "onnxruntime-gpu" if torch.cuda.is_available() else "onnxruntime",
        "tensorrt" if torch.cuda.is_available() else None,
        "openvino-dev",
    ]

    for package in packages:
        if package is None:
            continue
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
            print(f"✅ {package}")
        except Exception as e:
            print(f"⚠️ {package} 安裝失敗: {e}")

def export_yolov11_models(model_path: Path) -> Dict[str, bool]:
    """YOLOv11專用模型導出函數"""
    if not model_path.exists():
        print(f"❌ 模型文件不存在: {model_path}")
        return {}

    try:
        from ultralytics import YOLO
        model = YOLO(str(model_path))
        export_results = {}

        print("📦 開始導出YOLOv11模型...")

        # YOLOv11優化的ONNX導出
        print("\n📄 導出 ONNX 格式...")
        try:
            onnx_path = model.export(
                format='onnx',
                imgsz=1280,        # YOLOv11推薦解析度
                half=False,        # 保持FP32確保小物體檢測準確
                dynamic=True,      # 動態輸入尺寸
                simplify=True,
                opset=17,
                optimize=True,
                verbose=False
            )
            export_results['onnx'] = True
            print(f"✅ ONNX 成功: {onnx_path}")
        except Exception as e:
            print(f"❌ ONNX 失敗: {e}")
            export_results['onnx'] = False

        # YOLOv11 TensorRT導出 (A100專用)
        if torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0):
            print("\n🚀 A100 TensorRT 導出...")
            try:
                engine_path = model.export(
                    format='engine',
                    imgsz=1280,
                    half=True,        # YOLOv11 TensorRT支持更好的FP16
                    workspace=8,
                    device=0,
                    verbose=False
                )
                export_results['tensorrt'] = True
                print(f"✅ TensorRT 成功: {engine_path}")
            except Exception as e:
                print(f"❌ TensorRT 失敗: {e}")
                export_results['tensorrt'] = False

        # YOLOv11其他格式導出
        other_formats = [
            ('torchscript', 'TorchScript'),
            ('coreml', 'CoreML'),      # YOLOv11新增支持
            ('paddle', 'Paddle'),      # YOLOv11新增支持
        ]

        for fmt, name in other_formats:
            try:
                print(f"\n📄 導出 {name}...")
                path = model.export(format=fmt, imgsz=1280)
                export_results[fmt] = True
                print(f"✅ {name} 成功: {path}")
            except Exception as e:
                print(f"⚠️ {name} 跳過: {e}")
                export_results[fmt] = False

        return export_results

    except Exception as e:
        print(f"❌ 導出過程出錯: {e}")
        return {}


In [ ]:
def main_yolov11_smoking_detection():
    """YOLOv11香菸檢測主流程"""
    print("🚬 開始YOLOv11優化香菸檢測訓練")

    # 1. 安裝依賴
    install_yolov11_dependencies()

    # 2. 初始化
    trainer = YOLOv11SmokingTrainer(config)

    # 3. 環境設置
    if not trainer.setup_environment():
        return

    # 4. 執行訓練
    results = trainer.train()
    if results is None:
        return

    # 5. 模型驗證
    val_results = trainer.validate_model()

    # 6. 顯示訓練圖表
    results_png = config.PROJECT_PATH / config.EXPERIMENT_NAME / "results.png"
    if results_png.exists():
        img = Image.open(results_png)
        plt.figure(figsize=(15, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.title("🚬 YOLOv11 Smoking Detection Training Results")
        plt.show()

    # 7. 導出模型
    print("\n" + "="*50)
    print("📦 YOLOv11模型導出階段")
    print("="*50)

    export_results = export_yolov11_models(config.best_model_path)

    # 8. 最終總結
    print(f"\n📊 YOLOv11導出結果總結:")
    for format_name, success in export_results.items():
        status = "✅ 成功" if success else "❌ 失敗"
        print(f"   {format_name.upper()}: {status}")

    print("\n" + "="*60)
    print("🎉 YOLOv11香菸檢測訓練流程完成！")
    print("="*60)
    print(f"📈 驗證結果: {val_results}")
    print(f"📦 導出格式: {list(export_results.keys())}")
    print(f"💾 模型位置: {config.best_model_path}")
    print("="*60)

# 執行主程序

In [ ]:
if __name__ == "__main__":
    print("🎯 YOLOv11香菸檢測執行模式:")
    print("1. 完整訓練+導出: main_yolov11_smoking_detection()")
    print("2. 僅導出模型: quick_yolov11_export_only()")
    print("\n🚀 執行完整YOLOv11流程...")

    # 執行完整流程
    main_yolov11_smoking_detection()

🎯 YOLOv11香菸檢測執行模式:
1. 完整訓練+導出: main_yolov11_smoking_detection()
2. 僅導出模型: quick_yolov11_export_only()

🚀 執行完整YOLOv11流程...
🚬 開始YOLOv11優化香菸檢測訓練
📦 安裝YOLOv11必要套件...
✅ ultralytics>=8.3.0
✅ albumentations
✅ onnx>=1.16.0
✅ onnxruntime-gpu
✅ tensorrt
✅ openvino-dev
🚀 GPU資訊: NVIDIA A100-SXM4-80GB
💾 GPU記憶體: 85.2 GB
⚡ 檢測到A100 GPU，啟用YOLOv11專用優化

🛠️ 設置YOLOv11優化環境...
✅ 數據集解壓完成

🔍 分析小物體分佈...
📊 物體大小統計:
   平均大小: 0.047326
   最小大小: 0.000010
   最大大小: 1.000000
   小於0.01的比例: 39.70%
   小於0.001的比例: 7.19%
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🎯 開始YOLOv11優化訓練 - smoking_detect_v11m_ultra
Ultralytics 8.3.199 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, a

Traceback (most recent call last):
  File "/tmp/ipython-input-2386385740.py", line 189, in train
    results = self.model.train(**train_args)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 800, in train
    self.trainer.train()
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py", line 231, in train
    self._do_train(world_size)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py", line 510, in _do_train
    self.plot_metrics()
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py", line 747, in plot_metrics
    plot_results(file=self.csv, on_plot=self.on_plot)  # save results.png
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/utils/__init__.py", line 379, in wrapper
    result = func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python